# RealBasicVSR — Colab Master Setup

**Purpose:** Low-resolution video → 4× RealBasicVSR AI upscale → 1688×960 MP4.

This notebook is designed to be reused in a new Google Colab account/runtime.

**Known working environment**
- Python 3.10.x
- PyTorch 1.13.1+cu117
- torchvision 0.14.1
- mmcv-full 1.7.2
- mmedit 0.16.1
- openmim 0.3.9
- NumPy 1.26.4
- Tested successfully on Tesla T4

**Important:** GPU is required for the AI upscaling step. Final FFmpeg encoding does not require GPU.


## STEP 0 — Select a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU**. Then run the next cell.

In [ ]:
!nvidia-smi

Mon Sep 28 19:11:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## STEP 1 — Clone RealBasicVSR

In [ ]:
!git clone https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR

Cloning into '/content/RealBasicVSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 164 (delta 34), reused 23 (delta 23), pack-reused 113 (from 1)
Receiving objects: 100% (164/164), 25.97 MiB | 25.44 MiB/s, done.
Resolving deltas: 100% (50/50), done.


## STEP 2 — Create the known-working Python environment

Do not upgrade these packages to newer major versions during this workflow.

In [2]:
!python3.10 --version

Python 3.10.12


## STEP 3 — Verify versions and CUDA

Expected: NumPy 1.26.4, Torch 1.13.1+cu117, torchvision 0.14.1, MMCV 1.7.2, MMEdit 0.16.1, and CUDA available = True.

In [9]:
!/content/rbvsr_legacy/bin/pip install "numpy==1.26.4"

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [10]:
!/content/rbvsr_legacy/bin/pip install "opencv-python==4.8.1.78"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 5.0.0.93
    Uninstalling opencv-python-5.0.0.93:
      Successfully uninstalled opencv-python-5.0.0.93


In [11]:
!/content/rbvsr_legacy/bin/pip install "mmedit==0.16.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.8/606.8 KB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 106.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 KB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.9/333.9 KB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 25.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 82.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 KB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 KB 38.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 KB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 KB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 133.4 MB/s eta

In [13]:
!/content/rbvsr_legacy/bin/python -c "import torch, torchvision, numpy, mmcv, mmedit, cv2; print('Python:', __import__('sys').version); print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('Torch CUDA available:', torch.cuda.is_available()); print('Torchvision:', torchvision.__version__); print('NumPy:', numpy.__version__); print('MMCV:', mmcv.__version__); print('MMEditing:', mmedit.__version__); print('OpenCV:', cv2.__version__)"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Python: 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
Torch: 1.13.1+cu117
Torch CUDA: 11.7
Torch CUDA available: True
Torchvision: 0.14.1+cu117
NumPy: 1.26.4
MMCV: 1.7.2
MMEditing: 0.16.1
OpenCV: 4.8.1


In [15]:
!mkdir -p /content/RealBasicVSR/checkpoints

In [16]:
!gdown 1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

Downloading...
From (original): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID
From (redirected): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID&confirm=t&uuid=0d372f79-b6c9-4ec9-bdcc-4e7ef2443d4f
To: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
100% 210M/210M [00:03<00:00, 55.8MB/s]


In [17]:
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

-rw-r--r-- 1 root root 201M Nov 30  2021 /content/RealBasicVSR/checkpoints/RealBasicVSR.pth


In [35]:

from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving {Boruto}272.mp4 to {Boruto}272.mp4
Uploaded: ['{Boruto}272.mp4']


In [36]:
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,codec_name -show_entries format=duration -of default=noprint_wrappers=1 "/content/Boruto_272.mp4"

codec_name=h264
width=254
height=144
r_frame_rate=25/1
duration=30.000000


In [37]:

%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from mmedit.apis import init_model

CONFIG = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
CHECKPOINT = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"
INPUT = "/content/Boruto_272.mp4"
OUTPUT_DIR = "/content/rbvsr_frames"

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda:0"

print("Loading RealBasicVSR...")
model = init_model(CONFIG, CHECKPOINT, device=device)
model.eval()

cap = cv2.VideoCapture(INPUT)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Total frames:", total)
print("Upscaling...")

for n in tqdm(range(total)):
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(frame_rgb).permute(2, 0, 1).float() / 255.0
    img = img.unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img, test_mode=True)

    result = output[0, 0].clamp(0, 1)
    result = (result.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    result = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

    cv2.imwrite(f"{OUTPUT_DIR}/{n:05d}.png", result)

    del img, output, result
    torch.cuda.empty_cache()

cap.release()
print("DONE: frames saved to", OUTPUT_DIR)

Overwriting /content/RealBasicVSR/inference_frames_safe.py


In [38]:

!mkdir -p /content/RealBasicVSR/configs
!wget -q https://raw.githubusercontent.com/ckkelvinchan/RealBasicVSR/master/configs/realbasicvsr_x4.py -O /content/RealBasicVSR/configs/realbasicvsr_x4.py

In [39]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/test_output_structure.py

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-09-29 08:00:43,413 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
/content/rbvsr_legacy/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
OUTPUT TYPE: <class 'dict'>
OUTPUT KEYS: dict_keys(['lq', 

In [40]:

%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from mmedit.apis import init_model

CONFIG = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
CHECKPOINT = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"
INPUT = "/content/Boruto_272.mp4"
OUTPUT_DIR = "/content/rbvsr_frames"

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda:0"

print("Loading RealBasicVSR...")
model = init_model(CONFIG, CHECKPOINT, device=device)
model.eval()

cap = cv2.VideoCapture(INPUT)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Total frames:", total)
print("Upscaling...")

for n in tqdm(range(total)):
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(frame_rgb).permute(2, 0, 1).float() / 255.0
    img = img.unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        result_dict = model(img, test_mode=True)

    result = result_dict["output"]

    # Expected shape: [1, 1, 3, H, W]
    if result.dim() == 5:
        result = result[0, 0]
    elif result.dim() == 4:
        result = result[0]

    result = result.clamp(0, 1)
    result = (result.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    result = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

    cv2.imwrite(f"{OUTPUT_DIR}/{n:05d}.png", result)

    del img, result_dict, result
    torch.cuda.empty_cache()

cap.release()
print("DONE: frames saved to", OUTPUT_DIR)

Overwriting /content/RealBasicVSR/inference_frames_safe.py


In [41]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_frames_safe.py

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Loading RealBasicVSR...
2026-09-29 08:01:49,919 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
Total frames: 750
Upscaling...
  0% 0/750 [00:00<?, ?it/s]/content/rbvsr_legacy/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # typ

In [42]:
!find /content/rbvsr_frames -type f -name '*.png' | wc -l

750


In [52]:

!mkdir -p /content/rbvsr_results
!ffmpeg -y -framerate 25 -i /content/rbvsr_frames/%05d.png -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p /content/rbvsr_results/upscaled_video.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [53]:
!ffmpeg -y -i /content/rbvsr_results/upscaled_video.mp4 -i "/content/Boruto_272.mp4" -map 0:v:0 -map 1:a:0? -c:v copy -c:a copy -shortest /content/rbvsr_results/Boruto_272.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [56]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## STEP 4 — Check project files

In [57]:
!mkdir -p "/content/drive/MyDrive/RealBasicVSR_Output" && cp "/content/rbvsr_results/Boruto_272.mp4" "/content/drive/MyDrive/RealBasicVSR_Output/"

In [58]:
!ls -lh /content/RealBasicVSR/configs/realbasicvsr_x4.py
!ls -lh /content/RealBasicVSR/inference_realbasicvsr.py

-rw-r--r-- 1 root root 1.1K Sep 29 08:00 /content/RealBasicVSR/configs/realbasicvsr_x4.py
ls: cannot access '/content/RealBasicVSR/inference_realbasicvsr.py': No such file or directory


## STEP 5 — RealBasicVSR checkpoint

Place `RealBasicVSR.pth` at:

`/content/RealBasicVSR/checkpoints/RealBasicVSR.pth`

The checkpoint was previously verified successfully and was about 201 MB.


In [ ]:
!mkdir -p /content/RealBasicVSR/checkpoints
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
!/content/rbvsr_legacy/bin/python -c "import torch; p='/content/RealBasicVSR/checkpoints/RealBasicVSR.pth'; x=torch.load(p,map_location='cpu'); print(x.keys())"


## STEP 6 — Upload the input video

Use the upload button in the Colab Files panel, or run the next cell.

The example below expects:

`/content/Hindi Dub_BySHBD 240p.mp4`

If your filename is different, change `INPUT` in the later processing cell.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


### Check the uploaded video

In [ ]:
INPUT='/content/Hindi Dub_BySHBD 240p.mp4'
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,duration -of default=noprint_wrappers=1 "$INPUT"


## STEP 7 — Why this notebook uses one frame at a time

We tested multi-frame processing on a Tesla T4. A 5-frame batch caused CUDA out-of-memory, while single-frame inference succeeded.

Therefore this notebook uses:

**one input frame → one RealBasicVSR inference → immediately save output**

This is slower than large batches but is the proven low-memory method for this project.

The original approach also accumulated many output tensors in RAM. This notebook avoids that by saving every frame immediately.


## STEP 8 — Create the safe one-frame inference script

In [ ]:
%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os, cv2, torch, mmcv
from tqdm import tqdm
from mmcv import Config
from mmedit.models import build_model

INPUT='/content/Hindi Dub_BySHBD 240p.mp4'
OUT='/content/rbvsr_frames'
CONFIG='/content/RealBasicVSR/configs/realbasicvsr_x4.py'
CHECKPOINT='/content/RealBasicVSR/checkpoints/RealBasicVSR.pth'

os.makedirs(OUT, exist_ok=True)

cfg=Config.fromfile(CONFIG)
model=build_model(cfg.model, train_cfg=None, test_cfg=cfg.get('test_cfg'))
ck=torch.load(CHECKPOINT, map_location='cpu')
model.load_state_dict(ck['state_dict'], strict=False)
del ck
model=model.cuda().eval()

total=int(cv2.VideoCapture(INPUT).get(cv2.CAP_PROP_FRAME_COUNT))
cap=cv2.VideoCapture(INPUT)
print('Total frames:', total, flush=True)

for n in tqdm(range(total), desc='Upscaling', unit='frame'):
    ok, frame=cap.read()
    if not ok:
        print('Frame read failed:', n, flush=True)
        break

    img=torch.from_numpy(
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ).float().permute(2,0,1).unsqueeze(0).unsqueeze(0).cuda()/255.0

    with torch.no_grad():
        out=model(img, test_mode=True)['output']

    result=(
        out[0,0].clamp(0,1)
        .permute(1,2,0)
        .cpu().numpy()*255
    ).astype('uint8')

    result=cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

    cv2.imwrite(
        os.path.join(OUT, f'{n:05d}.png'),
        result
    )

    del img,out,result,frame
    torch.cuda.empty_cache()

cap.release()
print('DONE: frames saved to', OUT, flush=True)


## STEP 9 — Run the AI upscaling

Before a fresh run, clear incomplete frames. Then start processing. **Do not interrupt the cell while it is running.**

In [ ]:
!rm -rf /content/rbvsr_frames
!mkdir -p /content/rbvsr_frames
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_frames_safe.py


## STEP 10 — Verify the generated frames

For the known test video, the expected count is 485.

In [ ]:
!ls /content/rbvsr_frames | wc -l


## STEP 11 — Create the upscaled video (GPU NOT required)

Only run this after the expected frame count is present.

The output resolution for the 422×240 test video is 1688×960.


In [ ]:
!mkdir -p /content/rbvsr_results
!ffmpeg -y -framerate 25 -i /content/rbvsr_frames/%05d.png -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p /content/rbvsr_results/upscaled_video.mp4


## STEP 12 — Add the original audio

This copies the source AAC audio without re-encoding it.

In [ ]:
!ffmpeg -y -i /content/rbvsr_results/upscaled_video.mp4 -i "/content/Hindi Dub_BySHBD 240p.mp4" -map 0:v:0 -map 1:a:0? -c:v copy -c:a copy -shortest /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4


## STEP 13 — Verify the final MP4

In [ ]:
!ffprobe -v error -show_entries format=duration,size -show_entries stream=index,codec_name,width,height,r_frame_rate -of default=noprint_wrappers=1 /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4
!ls -lh /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4


## STEP 14 — Save the final output to Google Drive

`/content` is temporary. After a successful run, copy the final MP4 to Google Drive or download it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4 /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/Hindi_Dub_BySHBD_4x.mp4


# Troubleshooting / Known Lessons

### NumPy error
If you see:
`AttributeError: module 'numpy' has no attribute 'bool8'`

Use:
`numpy==1.26.4`

### Wrong max sequence argument
The repository's argument is:
`--max_seq_len`

not:
`--max-seq-len`

However, this master workflow does not depend on that argument because it uses the safe one-frame script.

### GPU unavailable
If:
`nvidia-smi: command not found`

the current runtime has no GPU. Do not start AI processing.

If Colab reports:
`Cannot connect to GPU backend`

GPU access is temporarily unavailable/limited. Try another eligible runtime/account or wait.

### Colab runtime reset
Files under `/content` can disappear after a reset. The 485 frames are temporary unless copied to persistent storage.

### Heredoc error
Avoid:
`!python - <<'PY'`

Use the `%%writefile` cell provided in this notebook.

### 5-frame CUDA OOM
Do not change the safe processor to large frame batches on a T4. Single-frame inference was the proven successful method.

# Known Successful Test

Input:
- 422×240
- 25 FPS
- 19.4 seconds
- 485 frames

GPU:
- Tesla T4

Successful environment:
- Python 3.10.12
- PyTorch 1.13.1+cu117
- torchvision 0.14.1
- mmcv-full 1.7.2
- mmedit 0.16.1
- openmim 0.3.9
- NumPy 1.26.4

Successful AI run:
- 485 frames
- 4× upscale
- 1688×960
- about 3m54s
- about 2.07 frames/sec

**END OF MASTER NOTEBOOK**
